# Introducción al reconocimiento de entidades nombradas (NER) con spaCy

**Curso:** Procesamiento de Lenguaje Natural · Unidad 4 — Etiquetado de secuencias y NER  
**Institución:** Universidad Tecnológica de Bolívar  
**Referencia:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3.ª ed.), Cap. 17 — Sequence Labeling for Parts of Speech and Named Entities  
**Compatibilidad con Python:** 3.10 · 3.11 · 3.12 · 3.13

## Contexto y Propósito

Este notebook introduce el **reconocimiento de entidades nombradas (NER)**: encontrar en un texto los nombres de personas, organizaciones y lugares, y clasificarlos. Usa el modelo preentrenado en español de **spaCy** sobre una oración de ejemplo, extrae sus entidades, las visualiza con displaCy y muestra el análisis lingüístico completo que spaCy hace de cada palabra (lema, categoría gramatical, dependencia sintáctica), además del árbol de dependencias.

**Objetivo:** conocer qué produce un *pipeline* de PLN preentrenado listo para usar, qué entidades reconoce (y cuáles no) y cómo inspeccionar sus resultados antes de integrarlo en una aplicación.

## Ejemplos de Aplicación en el Mundo Real

Cada caso muestra, paso a paso, cómo se aplica la técnica de este notebook a un problema real.

### Gobierno y legal — Personas y lugares en noticias

- **Problema:** una entidad quiere indexar las noticias por persona y lugar mencionados.
- **Entrada:** texto de las noticias.
- **Cómo se aplica el NER preentrenado de spaCy:**
  1. `es_core_news_sm` detecta PER, LOC y ORG en cada noticia.
  2. displaCy permite revisar rápido si las entidades son correctas.
  3. Los lemas agrupan variantes de una misma mención.
- **Salida:** índice de entidades por noticia.
- **Decisión:** búsqueda y alertas por persona o lugar.

### Finanzas y servicio al cliente — Entidades en noticias de mercado

- **Problema:** filtrar las noticias relevantes para un analista.
- **Entrada:** titulares económicos.
- **Cómo se aplica el NER preentrenado de spaCy:**
  1. NER identifica empresas (ORG) y lugares (LOC).
  2. El análisis de dependencias muestra qué hizo cada empresa (sujeto–verbo).
  3. Se descartan titulares sin empresas de interés.
- **Salida:** empresa, acción y lugar por titular.
- **Decisión:** tablero de noticias por empresa.

## Requisitos Previos

**Instalación** (una sola vez, desde la terminal; en Jupyter o Colab antepón `!`):

```bash
pip install spacy
python -m spacy download es_core_news_md
```

**Librerías que se importan:**

| Librería | Para qué se usa |
|---------|-----------------|
| `spacy` | Carga del modelo en español y procesamiento del texto |
| `spacy.displacy` | Visualización de entidades y del árbol de dependencias en Jupyter |

**Modelo:** `es_core_news_md` (~40 MB), entrenado con noticias en español.

## Marco metodológico: pipeline ML / LLM (4×3)

![Pipeline ML y LLM](img/Pipeline_ML_LLM_4x3.png)

El diagrama tiene una franja común de datos (arriba), dos rutas de modelado y una franja de operación (abajo): **ruta clásica** (*feature-based ML*) y **ruta LLM** (*foundation models*). Este notebook recorre la **ruta clásica**. Etapas del diagrama que cubre:

| Etapa del diagrama | Dónde se aborda en este notebook |
|--------------------|----------------------------------|
| Data | Sección 3: oración de ejemplo |
| Data Preprocessing | Sección 6: tokenización, lemas y categorías gramaticales que produce el pipeline |
| Pre-trained Models | Sección 2: modelo `es_core_news_md` de spaCy |
| Predictions | Secciones 4, 5 y 7: entidades, visualización y árbol de dependencias |

El modelo de spaCy es un modelo estadístico preentrenado (red neuronal pequeña, no un LLM), por eso se ubica en la ruta clásica, en la caja **Pre-trained Models**. Aquí solo se usa para predecir; ajustarlo con datos propios está en `training_ner.ipynb`.

## Estándares de referencia

Prácticas de este notebook alineadas con estándares de IEEE SA, ISO/IEC y la Open Source Initiative (OSI). Se citan como guía de buenas prácticas, no como certificación de cumplimiento.

| Estándar | Cómo se aplica aquí |
|----------|---------------------|
| **IEEE 7001-2021** — Transparencia de sistemas autónomos | displaCy y la tabla de atributos por token hacen visibles las decisiones del modelo, para revisarlas antes de usarlas. |
| **IEEE 3119-2025** — Adquisición (*procurement*) de IA y sistemas de decisión automatizada | Usar un modelo preentrenado es adquirir IA de terceros: hay que conocer sus etiquetas, su dominio de entrenamiento (noticias) y sus límites. |
| **IEEE 7003-2024** — Consideraciones sobre sesgo algorítmico | Un modelo entrenado con noticias puede rendir peor con otros registros (redes sociales, textos clínicos, español regional). |
| **OSI · Open Source AI Definition 1.0 (2024)** — Qué hace que un sistema de IA sea de código abierto: libertades de usar, estudiar, modificar y compartir, con acceso a código, parámetros e información de los datos | spaCy y sus modelos en español se distribuyen con licencias abiertas e información de sus corpus de entrenamiento. |

Este notebook introduce el reconocimiento de entidades nombradas (NER) con el modelo de spaCy para español. Se procesa una oración de ejemplo, se extraen sus entidades (personas, organizaciones, lugares), se visualizan con displaCy y se inspecciona el análisis lingüístico completo de cada token.

## 1. Configuración: importar librerías

In [1]:
# spaCy y su visualizador
import spacy
from spacy import displacy

## 2. Cargar el modelo de PLN en español

Usamos `es_core_news_md` (modelo mediano en español entrenado con textos de noticias). Si aún no está instalado, ejecuta una vez:
```
python -m spacy download es_core_news_md
```

In [2]:
# Carga el pipeline completo: tokenizador, etiquetador, analizador de dependencias, NER…
nlp = spacy.load("es_core_news_md")

## 3. Procesar una oración de ejemplo

Usamos una oración en español de estilo periodístico, el mismo tipo de texto con el que se entrenó el modelo.

In [3]:
# Oración de ejemplo para la demostración de NER
texto = (
    "El presidente Pedro Sánchez visitó la sede de Naciones Unidas "
    "en Nueva York el 20 de septiembre."
)

In [4]:
# Un solo paso por el pipeline produce todas las anotaciones
doc = nlp(texto)

## 4. Extraer las entidades nombradas

In [6]:
# Cada entidad: texto y etiqueta
for ent in doc.ents:
    print(ent.text, ent.label_)

Pedro Sánchez PER
Naciones Unidas ORG
Nueva York LOC


> **Interpretación de la salida:** cada línea muestra una entidad y su etiqueta: `PER` = persona (*Pedro Sánchez*), `ORG` = organización (*Naciones Unidas*), `LOC` = lugar (*Nueva York*). La fecha *20 de septiembre* **no** aparece: los modelos de spaCy para español solo reconocen cuatro tipos (la celda siguiente lo verifica). Para fechas, montos o tipos propios hay que entrenar un NER (ver `training_ner.ipynb`).

In [ ]:
# Tipos de entidad que reconoce el modelo
print(nlp.get_pipe("ner").labels)

> **Interpretación de la salida:** el modelo solo conoce `LOC`, `MISC`, `ORG` y `PER`. `MISC` agrupa entidades que no son persona, lugar ni organización (eventos, obras, nacionalidades).

## 5. Visualizar las entidades con displaCy

In [7]:
# Resalta las entidades dentro del texto, con un color por tipo
displacy.render(doc, style="ent", jupyter=True)

> **Interpretación de la salida:** la oración se muestra con cada entidad resaltada con un color según su tipo. Es una forma rápida de verificar visualmente las predicciones antes de integrar el modelo en un sistema.

## 6. Análisis lingüístico por token

spaCy anota cada token con su categoría gramatical, relación de dependencia, lema y rasgos de forma, todo en un solo paso por el pipeline.

In [8]:
# Por token: texto, lema, categoría universal, etiqueta fina, dependencia, forma, ¿alfabético?, ¿stopword?
for token in doc:
    print(
        token.text,
        token.lemma_,
        token.pos_,
        token.tag_,
        token.dep_,
        token.shape_,
        token.is_alpha,
        token.is_stop,
    )

El el DET DET det Xx True True
presidente presidente NOUN NOUN nsubj xxxx True False
Pedro Pedro PROPN PROPN appos Xxxxx True False
Sánchez Sánchez PROPN PROPN flat Xxxxx True False
visitó visitar VERB VERB ROOT xxxx True False
la el DET DET det xx True True
sede sede NOUN NOUN obj xxxx True False
de de ADP ADP case xx True True
Naciones Naciones PROPN PROPN nmod Xxxxx True False
Unidas Unidas PROPN PROPN flat Xxxxx True False
en en ADP ADP case xx True True
Nueva Nueva PROPN PROPN nmod Xxxxx True True
York York PROPN PROPN flat Xxxx True False
el el DET DET det xx True True
20 20 NUM NUM obl dd False False
de de ADP ADP case xx True True
septiembre septiembre NOUN NOUN compound xxxx True False
. . PUNCT PUNCT punct . False False


> **Interpretación de la salida:** cada fila es un token con: forma, lema (forma base: *visitó* → *visitar*), categoría gramatical universal, etiqueta fina, relación de dependencia sintáctica, patrón de forma (`Xxxxx` = inicial mayúscula), si es alfabético y si es *stopword*. Estas anotaciones sirven para búsqueda por lemas, análisis sintáctico y extracción de rasgos para modelos de ML.

## 7. Visualizar el árbol de dependencias

El análisis de dependencias muestra las relaciones gramaticales entre tokens (sujeto, objeto, modificador, etc.). En Jupyter se usa `displacy.render`; `displacy.serve` iniciaría un servidor web aparte y bloquearía la ejecución.

In [9]:
# Árbol de dependencias sintácticas
displacy.render(doc, style="dep", jupyter=True)

> **Interpretación de la salida:** el diagrama muestra de qué palabra depende cada token. El verbo raíz *visitó* gobierna al sujeto *presidente* y al objeto *sede*; los nombres de varias palabras (*Pedro Sánchez*, *Naciones Unidas*, *Nueva York*) se unen con la relación `flat`.